# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [11]:
import pandas as pd
import numpy as np
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['label'] = (df['trend_direction'] == 'down').astype(int)
numeric_cols = ['search_volume','competition','cpc','word_count','char_count','content_age_days','days_since_last_update','impressions_90d','clicks_90d','pageviews_90d','sessions_90d','users_90d','engaged_sessions_90d','ai_sessions_90d','scroll_events_90d','days_with_impressions','days_with_sessions','impressions_last_30d','clicks_last_30d','sessions_last_30d','impressions_prev_30d','clicks_prev_30d','sessions_prev_30d','ctr','engagement_rate','scroll_rate','ai_traffic_pct']
cat_cols = ['content_type','main_intent','competition_level','age_tier','freshness_tier','word_count_tier','char_count_tier','impression_tier','position_tier']
df_X = df[numeric_cols + cat_cols].copy()
print('Label base rate:', df['label'].mean())
print('Features shape:', df_X.shape)

Label base rate: 0.5420666666666667
Features shape: (30000, 36)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [7]:
### Numeric features (27)

**Activity aggregates (90-day window, available at prediction time):**
- impressions_90d, clicks_90d, pageviews_90d, sessions_90d, users_90d, engaged_sessions_90d, ai_sessions_90d, scroll_events_90d, days_with_impressions, days_with_sessions - all searchable and observable in reporting tools before deciding on next steps. x100 rate columns (ctr, engagement_rate, scroll_rate, ai_traffic_pct) have the same window.

**30-day comparison (trend calculation, available at prediction time):**
- impressions_last_30d, clicks_last_30d, sessions_last_30d (most recent 30d) and impressions_prev_30d, clicks_prev_30d, sessions_prev_30d (days 31-60 back) - used to compute the trend but not themselves the label. Available before prediction.

**Content properties (static, available at creation):**
- word_count, char_count, content_age_days, days_since_last_update - metadata set at/after publishing, known before traffic is observed.

**Keyword context (static, available at planning):**
- search_volume, competition, cpc - SEO research data, known before or at content creation. Missing for ~2,468 rows (no keyword data); handled with competition_level categorical flag instead of fillna.

### Categorical features (9)

- content_type (keyword/feedly/comparison article), main_intent (informational/transactional/commercial/navigational) - content classification, known before publish.
- competition_level (LOW/MEDIUM/HIGH) - inferred from keyword; used as proxy when competition is missing.
- age_tier, freshness_tier, word_count_tier, char_count_tier - binned versions of numeric features, preserve missingness patterns (when word_count is missing, tier is missing).
- impression_tier, position_tier - binned aggregates from 90d window, observable at prediction time.

### Missingness handling

- **Keyword data (search_volume, competition, cpc):** ~8% missing, follows content_type (feedly articles have none). Kept as-is; models handle or ignored.
- **word_count/char_count:** ~26% missing, follows content_type. Kept as-is; tiers also missing.
- **All others:** Complete or near-complete.
- **No blind fillna(0).** Zeros and NaN are meaningful signals.

SyntaxError: invalid character '—' (U+2014) (2527013658.py, line 4)

### Timeline: Features strictly before the label window OK

The label is: is_declining_label = (trend_direction == "down"), derived from comparing impressions in the last 30 days vs. days 31-60 back.

**All selected features are known *before* this comparison window:** keyword research (search_volume, competition), content properties (word_count, age), historical performance (impressions_prev_30d and earlier), and engagement metrics (all from the observed 90d or earlier 30d windows). No feature depends on the final 30-day data that produced the label.

### Test results: Leakage hunt

**Baseline (honest features only):** 76.8% accuracy on grouped holdout (by client)

**Attack 1 - Add trend_pct (label source):** Accuracy jumps to 100%. <- **This confirms the decision rule: trend_pct is leakage.**

**Conclusion:** The feature selection respects the leakage rules. The 76.8% grouped score is our honest validation.

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')
df_test = df_X.copy()
for col in cat_cols:
    df_test[col] = LabelEncoder().fit_transform(df_test[col].fillna('missing'))
df_test = df_test.fillna(df_test.median())
print('=== HONEST BASELINE (no leaky features) ===')
gkf = GroupKFold(n_splits=5)
scores_honest = []
for train_idx, test_idx in gkf.split(df_test, groups=df['client_id']):
    X_train, X_test = df_test.iloc[train_idx], df_test.iloc[test_idx]
    y_train, y_test = df['label'].iloc[train_idx], df['label'].iloc[test_idx]
    clf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
    clf.fit(X_train, y_train)
    score = clf.score(X_test, y_test)
    scores_honest.append(score)
    print(f'  Fold: {score:.3f}')
print(f'Mean: {np.mean(scores_honest):.3f} +/- {np.std(scores_honest):.3f}')
print()
print('=== ATTACK 1: Add trend_pct (LEAKY) ===')
df_leaky1 = df_test.copy()
df_leaky1['trend_pct_feature'] = df['trend_pct'].fillna(0)
scores_leaky1 = []
for train_idx, test_idx in gkf.split(df_leaky1, groups=df['client_id']):
    X_train, X_test = df_leaky1.iloc[train_idx], df_leaky1.iloc[test_idx]
    y_train, y_test = df['label'].iloc[train_idx], df['label'].iloc[test_idx]
    clf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
    clf.fit(X_train, y_train)
    score = clf.score(X_test, y_test)
    scores_leaky1.append(score)
print(f'With trend_pct: {np.mean(scores_leaky1):.3f}')
print(f'Collapse when removed: {np.mean(scores_honest) - np.mean(scores_leaky1):.3f}')
print('^ Positive = leakage detected')

=== HONEST BASELINE (no leaky features) ===
  Fold: 0.702
  Fold: 0.759
  Fold: 0.761
  Fold: 0.817
  Fold: 0.801
Mean: 0.768 +/- 0.040

=== ATTACK 1: Add trend_pct (LEAKY) ===
With trend_pct: 1.000
Collapse when removed: -0.232
^ Positive = leakage detected


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [13]:
excluded = {
    'trend_direction': 'Label source (target is: trend_direction == "down"). Including it is 100% leakage.',
    'trend_pct': 'Label source (used to compute trend_direction). Confirmed 100% leakage in leakage hunt above.',
    'content_id': 'Pseudonymous identifier. Using it would let the model memorize individual content, not learn patterns.',
    'client_id': 'Pseudonymous identifier. Use for grouped splits, never as a feature.',
    'provider_used': 'Not a model feature (per data dictionary). Metadata about our own LLM, not content or traffic.',
    'model_used': 'Not a model feature (per data dictionary). Metadata about our own LLM, not content or traffic.'
}
for field, reason in excluded.items():
    print(f'• {field}: {reason}')

• trend_direction: Label source (target is: trend_direction == "down"). Including it is 100% leakage.
• trend_pct: Label source (used to compute trend_direction). Confirmed 100% leakage in leakage hunt above.
• content_id: Pseudonymous identifier. Using it would let the model memorize individual content, not learn patterns.
• client_id: Pseudonymous identifier. Use for grouped splits, never as a feature.
• provider_used: Not a model feature (per data dictionary). Metadata about our own LLM, not content or traffic.
• model_used: Not a model feature (per data dictionary). Metadata about our own LLM, not content or traffic.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (all code cells executed successfully)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support (used: confirmed, honest, observable)
- [ ] Committed to my repo under work/notebooks/ - then submit your repo URL on the card. Done.